# XGBoost

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd

import xgboost as xgb

In [2]:
SEED = 10

In [3]:
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

## Dataset

In [4]:
from dataset_NCT00981058 import DatasetNCT00981058

ds_name = 'real_NCT00981058_inject'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
project = 'SurvSurfBenchmark_NCT00981058_inject'
SAVE_PATH = f'./xgboost_models/{project}/xgboost_mono_{SEED}'
g_resol = 1
t_resol = 7
split = 'train'
ds = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)
df_train = ds._get_df_Xy_trans_obs()

split = 'train'
ds = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid_naless', 
    separate_g_from_feats=False
)
df_train_pred = ds._get_df_Xy_true_prob(dropna=True)

split = 'val'
ds = DatasetNCT00981058(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    split=split, 
    mode='true_probs_grid_naless', 
    separate_g_from_feats=False
)
df_val = ds._get_df_Xy_true_prob(dropna=True)

In [5]:
df_train.head()

,subject,event_observed,duration,g_max_by_time,feat__AGE,feat__BLOOD AND LYMPHATIC SYSTEM DISORDERS,feat__BMIBL,feat__BSABL,feat__Body Surface Area (m^2),feat__CARDIAC DISORDERS,...,feat__SOCIAL CIRCUMSTANCES,feat__SURGICAL AND MEDICAL PROCEDURES,feat__Systolic Blood Pressure (mmHg),feat__Temperature (°C),feat__VASCULAR DISORDERS,"feat__WBCCAT1_> 11,000 µl (11 x 10^9/L)",feat__Weight (kg),feat__traj_clust,weight,is_t_trans
0,b'001-7882',1,2.0,2.0,-0.712101,0.666667,-0.911691,0.110984,0.110984,0.0,...,0.0,0.666667,-1.685565,-0.746404,0.0,0.0,-0.381528,0.25,1,1
1,b'001-7882',1,15.0,4.0,-0.712101,0.666667,-0.911691,0.110984,0.110984,0.0,...,0.0,0.666667,-1.685565,-0.746404,0.0,0.0,-0.381528,0.25,1,1
2,b'001-7882',0,126.0,5.0,-0.712101,0.666667,-0.911691,0.110984,0.110984,0.0,...,0.0,0.666667,-1.685565,-0.746404,0.0,0.0,-0.381528,0.25,1,1
3,b'008-7881',1,4.0,1.0,-0.347338,0.000000,0.000045,-0.032476,-0.032476,0.0,...,0.0,0.000000,-1.361689,-1.580505,0.0,1.0,-0.100561,0.50,1,1
4,b'008-7881',0,62.0,2.0,-0.347338,0.000000,0.000045,-0.032476,-0.032476,0.0,...,0.0,0.000000,-1.361689,-1.580505,0.0,1.0,-0.100561,0.50,1,1


## Feature transforms

In [6]:
x_train = df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].astype('float32')
x_train['g_max_by_time'] = x_train['g_max_by_time']/ds.g_max
mono_constraints = {i:0 if i != 'g_max_by_time' else -1 for i in x_train.columns}
x_train = x_train


x_train_pred = df_train_pred.loc[:,df_train_pred.columns.str.startswith('feat')|df_train_pred.columns.str.startswith('g_max_by_time')].astype('float32')
x_train_pred['g_max_by_time'] = x_train_pred['g_max_by_time']/ds.g_max
x_train_pred = x_train_pred


x_val = df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].astype('float32')
x_val['g_max_by_time'] = x_val['g_max_by_time']/ds.g_max
x_val = x_val

assert all(df_train.loc[:,df_train.columns.str.startswith('feat')|df_train.columns.str.startswith('g_max_by_time')].columns == (
    df_val.loc[:,df_val.columns.str.startswith('feat')|df_val.columns.str.startswith('g_max_by_time')].columns
))



In [7]:
import xgboost as xgb
y_train = df_train['duration']*np.where(df_train['event_observed']==1, 1, -1)


params = {
    'objective': 'survival:cox',
    'monotone_constraints': mono_constraints,
    'random_state':10
}
dtrain = xgb.DMatrix(x_train, label=y_train)
xgboost_model = xgb.train(params, dtrain)
from lifelines import CoxPHFitter

train_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain)),
    'time': df_train['duration'],
    'event': df_train['event_observed']
})

# Fit a simple Cox model on the XGBoost scores to get the baseline hazard (a quick work-around re-implementing Breslow)
cph = CoxPHFitter()
cph.fit(train_results, duration_col='time', event_col='event')


<lifelines.CoxPHFitter: fitted with 846 total observations, 321 right-censored observations>

## Prediction

In [8]:
df_val['event_observed'].value_counts()

event_observed
1.0    36877
0.0    26660
Name: count, dtype: int64

In [9]:
df_train_pred['event_observed'].value_counts()

event_observed
1.0    172529
0.0     89400
Name: count, dtype: int64

In [10]:
dval = xgb.DMatrix(x_val)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

val_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dval)),
})
pred_grid_val = 1-cph.predict_survival_function(val_results).T



In [11]:
from sklearn.metrics import mean_squared_error
df_val_pred = df_val.copy()
df_val_pred['pred'] = [np.interp(x=t, xp=pred_grid_val.columns, fp=pred_grid_val.loc[idx,:]) for idx, t in df_val_pred['duration'].items()]
mean_squared_error(y_true=df_val_pred['event_observed'], y_pred=df_val_pred['pred'])

0.07506969978922477

In [12]:
dtrain_pred = xgb.DMatrix(x_train_pred)
# dtrain = xgb.DMatrix(x_train, label=y_train)
from lifelines import CoxPHFitter

train_pred_results = pd.DataFrame({
    'xgb_risk': np.log(xgboost_model.predict(dtrain_pred)),
})
pred_grid_train = 1-cph.predict_survival_function(train_pred_results).T

In [13]:
df_train_pred_all = df_train_pred.copy()
df_train_pred_all['pred'] = [np.interp(x=t, xp=pred_grid_train.columns, fp=pred_grid_train.loc[idx,:]) for idx, t in df_train_pred_all['duration'].items()]
mean_squared_error(y_true=df_train_pred_all['event_observed'], y_pred=df_train_pred_all['pred'])

0.044351501687292946

In [14]:
import pickle
xgboost_model.save_model(f'{SAVE_PATH}_trees.ubj')


with open(f'{SAVE_PATH}_cph.pkl', 'wb') as fp:
    pickle.dump(cph, file=fp)